# Colab Train And Submit

Simple Colab workflow: load data, run baseline, train LoRA, package `submission.zip`.

All settings are in the next cell.

## 1. Config

In [ ]:
from pathlib import Path

# Possible MODEL_NAME values:
# "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16" # challenge target
# "google/gemma-4-E2B-it"                      # smaller test
# "google/gemma-4-E4B-it"                      # heavier Gemma test
MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

PROJECT_ROOT = Path("/content/nemotron_challenge")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
ZIP_PATH = Path("/content/nvidia-nemotron-model-reasoning-challenge.zip")
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "colab_lora"
ADAPTER_DIR = OUTPUT_DIR / "adapter"
SANITY_CSV_PATH = OUTPUT_DIR / "sanity_test_predictions.csv"
SUBMISSION_ZIP_PATH = PROJECT_ROOT / "submission.zip"

# Start small for Nemotron training. Raise to 512/1024 after one step works.
MAX_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 32
# Use an integer for a partial run, or None to use all rows not in validation.
# Keep this small until the BF16 MoE patch reaches a few training steps.
TRAIN_ROWS = 1024
EVAL_ROWS = 256
BASELINE_ROWS = 3

SYSTEM_PROMPT = "Solve the puzzle. Put only the final answer inside \\boxed{} with no trailing text."

# Kaggle submission adapter rank must be at most 32.
LORA_R = 4
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
# For Nemotron use ["in_proj", "out_proj"]. For Gemma use ["q_proj", "v_proj"].
LORA_TARGET_MODULES = ["in_proj", "out_proj"]
# Nemotron Mamba activations are memory-heavy. Keep micro-batch at 1.
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 32
EPOCHS = 1
LEARNING_RATE = 3e-4

# "bf16_fast" is faster/lower memory on A100/H100.
# "fp32_safe" is slower/heavier, but avoids BF16 custom-code dtype issues.
PRECISION_MODE = "bf16_fast"

# Required for Nemotron BF16 training after the index_add dtype error in doc/errors.md.
# This patches only the MoE temporary tensor dtype, not the whole model.
PATCH_NEMOTRON_MOE_DTYPE = True


## 2. Install and import

In [ ]:
!pip install packaging ninja
!pip install -q mamba-ssm --no-build-isolation
!pip uninstall -y -q causal-conv1d
# For any HF basic activities like loading models
# and tokenizers for running inference
# upgrade is a must for the newest Gemma model
!pip install --upgrade datasets
!pip install --upgrade transformers
!pip install --upgrade torchao

# For doing efficient stuff - PEFT
!pip install --upgrade peft
!pip install --upgrade trl
!pip install bitsandbytes
!pip install accelerate

# for logging and visualizing training progress
!pip install tensorboard


In [ ]:

import json
import re
import time
import types
import zipfile

import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from google.colab import files, userdata

HF_TOKEN = userdata.get("hftoken")


## 3. Load data

In [ ]:
if not (RAW_DIR / "train.csv").exists() or not (RAW_DIR / "test.csv").exists():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(RAW_DIR)

train = pd.read_csv(RAW_DIR / "train.csv")
test = pd.read_csv(RAW_DIR / "test.csv")

assert list(train.columns) == ["id", "prompt", "answer"]
assert list(test.columns) == ["id", "prompt"]
print("train:", train.shape, "test:", test.shape)
display(test)


## 4. Prepare data

In [ ]:
def normalize_answer(value) -> str:
    text = "" if value is None else str(value).strip()
    text = re.sub(r"^`+|`+$", "", text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip().strip(".")


def extract_final_answer(text: str) -> str:
    """Extract a final answer using Kaggle's boxed-answer priority.

    The official metric first looks for answers in ``\\boxed{...}``. A
    literal ``}`` can be part of some answers, so this uses the last closing
    brace before the next boxed answer/end of text instead of stopping at the
    first ``}``.
    """
    text = "" if text is None else str(text)
    boxed_starts = list(re.finditer(r"\\boxed\{", text))
    if boxed_starts:
        matches = []
        for idx, match in enumerate(boxed_starts):
            start = match.end()
            end = boxed_starts[idx + 1].start() if idx + 1 < len(boxed_starts) else len(text)
            segment = text[start:end]
            last_brace = segment.rfind("}")
            matches.append(segment[:last_brace] if last_brace != -1 else segment)
        non_empty = [match.strip() for match in matches if match.strip()]
        return normalize_answer(non_empty[-1] if non_empty else matches[-1])

    patterns = [
        r"The final answer is:\s*([^\n]+)",
        r"Final answer is:\s*([^\n]+)",
        r"Final answer\s*[:：]\s*([^\n]+)",
        r"final answer\s*[:：]\s*([^\n]+)",
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return normalize_answer(matches[-1])

    number_matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if number_matches:
        return normalize_answer(number_matches[-1])

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in reversed(lines):
        line = re.sub(r"^(final answer|answer|assistant)\s*[:\-]\s*", "", line, flags=re.I).strip()
        if line:
            return normalize_answer(line.strip("`").strip())
    return normalize_answer(text)


def format_training_answer(answer: str) -> str:
    return f"\\boxed{{{normalize_answer(answer)}}}"


def build_sft_text(prompt: str, answer: str) -> str:
    return f"System:\n{SYSTEM_PROMPT}\n\nUser:\n{prompt}\n\nAssistant:\n{format_training_answer(answer)}"


def build_prompt(prompt: str) -> str:
    return f"System:\n{SYSTEM_PROMPT}\n\nUser:\n{prompt}\n\nAssistant:\n"


sft = train[["id", "prompt", "answer"]].copy()
sft["text"] = [build_sft_text(p, a) for p, a in zip(sft["prompt"], sft["answer"])]
valid = sft.sample(n=min(EVAL_ROWS, len(sft)), random_state=42)
train_sft = sft.drop(index=valid.index)
if TRAIN_ROWS is not None:
    train_sft = train_sft.head(TRAIN_ROWS)
baseline_questions = sft.head(BASELINE_ROWS).reset_index(drop=True)

train_ds = Dataset.from_pandas(train_sft[["text"]], preserve_index=False)
valid_ds = Dataset.from_pandas(valid[["text"]], preserve_index=False)
print("train rows:", len(train_ds), "valid rows:", len(valid_ds))


## 5. Load model

In [ ]:
bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
if PRECISION_MODE not in {"bf16_fast", "fp32_safe"}:
    raise ValueError("PRECISION_MODE must be 'bf16_fast' or 'fp32_safe'")
if PRECISION_MODE == "bf16_fast" and not bf16_supported:
    raise RuntimeError("This runtime does not support BF16. Use an A100/H100 or set PRECISION_MODE='fp32_safe'.")

compute_dtype = torch.bfloat16 if PRECISION_MODE == "bf16_fast" else torch.float32
model_dtype = torch.bfloat16 if bf16_supported else torch.float16
trainer_bf16 = PRECISION_MODE == "bf16_fast"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print("model:", MODEL_NAME)
print("precision mode:", PRECISION_MODE)
print("model dtype:", model_dtype)
print("compute dtype:", compute_dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=model_dtype,
    device_map={"": 0},
    trust_remote_code=True,
    token=HF_TOKEN,
)
model.config.use_cache = False
model.generation_config.use_cache = False
model.eval()


def _patched_nemotron_moe(self, hidden_states, topk_indices, topk_weights):
    final_hidden_states = torch.zeros_like(hidden_states, dtype=topk_weights.dtype)
    expert_mask = torch.nn.functional.one_hot(topk_indices, num_classes=len(self.experts))
    expert_mask = expert_mask.permute(2, 0, 1)

    for expert_idx in range(len(self.experts)):
        expert = self.experts[expert_idx]
        mask = expert_mask[expert_idx]
        token_indices, weight_indices = torch.where(mask)

        if token_indices.numel() > 0:
            expert_weights = topk_weights[token_indices, weight_indices]
            expert_input = hidden_states[token_indices]
            expert_output = expert(expert_input)
            weighted_output = expert_output * expert_weights.unsqueeze(-1)
            weighted_output = weighted_output.to(final_hidden_states.dtype)
            final_hidden_states.index_add_(0, token_indices, weighted_output)
        else:
            expert_dtype = expert.down_proj.weight.dtype
            dummy_out = expert(torch.zeros_like(hidden_states[0]).unsqueeze(0).to(expert_dtype))
            final_hidden_states = final_hidden_states + dummy_out.to(final_hidden_states.dtype)

    return final_hidden_states.type(hidden_states.dtype)


def patch_nemotron_moe_dtype(current_model):
    patched = 0
    for module in current_model.modules():
        if module.__class__.__name__ == "NemotronHMOE":
            module.moe = types.MethodType(_patched_nemotron_moe, module)
            patched += 1
    print("patched Nemotron MoE dtype modules:", patched)


if PRECISION_MODE == "bf16_fast" and PATCH_NEMOTRON_MOE_DTYPE:
    patch_nemotron_moe_dtype(model)


## 6. Module summary

In [ ]:
def dtype_name(dtype):
    return str(dtype).replace("torch.", "") if dtype is not None else "-"


module_rows = []
for name, module in model.named_modules():
    class_name = module.__class__.__name__
    if "Linear" in class_name or "Conv" in class_name:
        short_name = name.split(".")[-1]
        weight = getattr(module, "weight", None)
        compute_dtype = getattr(module, "compute_dtype", None)
        param_dtypes = sorted({dtype_name(p.dtype) for p in module.parameters(recurse=False)})
        buffer_dtypes = sorted({dtype_name(b.dtype) for b in module.buffers(recurse=False)})
        module_rows.append({
            "module": short_name,
            "class": class_name,
            "weight_dtype": dtype_name(getattr(weight, "dtype", None)),
            "compute_dtype": dtype_name(compute_dtype),
            "param_dtypes": ", ".join(param_dtypes) or "-",
            "buffer_dtypes": ", ".join(buffer_dtypes) or "-",
        })

module_summary = (
    pd.DataFrame(module_rows)
    .groupby(["module", "class", "weight_dtype", "compute_dtype", "param_dtypes", "buffer_dtypes"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(module_summary)
print("LoRA targets:", LORA_TARGET_MODULES)


## 7. Baseline inference

In [ ]:
def generate_answers(current_model, rows):
    records = []
    current_model.eval()
    for row in rows.itertuples(index=False):
        inputs = tokenizer(build_prompt(row.prompt), return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(current_model.device)
        start = time.time()
        with torch.no_grad():
            output_ids = current_model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        seconds = time.time() - start
        generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
        raw = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        records.append({"id": row.id, "answer": extract_final_answer(raw), "seconds": seconds, "raw_output": raw})
    return pd.DataFrame(records)

baseline_results = generate_answers(model, baseline_questions)
baseline_results["gold"] = baseline_questions["answer"]
display(baseline_results[["id", "gold", "answer", "seconds"]])


## 8. Train LoRA

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)
if LORA_R > 32:
    raise ValueError("Kaggle submissions require LoRA rank <= 32")

import gc
gc.collect()
torch.cuda.empty_cache()

args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    report_to="tensorboard",
    fp16=False,
    bf16=trainer_bf16,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    peft_config=lora_config,
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model(str(ADAPTER_DIR))


## 9. Compare after training

In [ ]:
after_results = generate_answers(trainer.model, baseline_questions)

comparison = baseline_results[["id", "gold", "answer"]].rename(columns={"answer": "before"})
comparison = comparison.merge(
    after_results[["id", "answer"]].rename(columns={"answer": "after"}),
    on="id",
)
comparison["before_match"] = comparison["gold"].map(normalize_answer) == comparison["before"].map(normalize_answer)
comparison["after_match"] = comparison["gold"].map(normalize_answer) == comparison["after"].map(normalize_answer)

display(comparison[["id", "gold", "before", "before_match", "after", "after_match"]])


## 10. Sanity predictions

In [ ]:
sanity_predictions = generate_answers(trainer.model, test)
sanity_submission = sanity_predictions[["id", "answer"]].copy()

assert list(sanity_submission.columns) == ["id", "answer"]
assert sanity_submission["id"].tolist() == test["id"].tolist()
assert sanity_submission["answer"].astype(str).str.len().gt(0).all()

SANITY_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
sanity_submission.to_csv(SANITY_CSV_PATH, index=False)
print("wrote sanity CSV:", SANITY_CSV_PATH)
display(sanity_submission)


## 11. Package adapter submission

In [ ]:
def validate_adapter_dir(adapter_dir: Path) -> list[Path]:
    required = ["adapter_config.json", "adapter_model.safetensors"]
    missing = [name for name in required if not (adapter_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(f"missing required adapter files in {adapter_dir}: {missing}")

    config = json.loads((adapter_dir / "adapter_config.json").read_text(encoding="utf-8"))
    rank = int(config.get("r", 0))
    if not 1 <= rank <= 32:
        raise ValueError(f"Kaggle submissions require LoRA rank <= 32; found r={rank}")
    print("adapter rank:", rank)
    return [adapter_dir / name for name in required]


def write_submission_zip(adapter_dir: Path, output_zip: Path) -> None:
    adapter_files = validate_adapter_dir(adapter_dir)
    output_zip.parent.mkdir(parents=True, exist_ok=True)
    if output_zip.exists():
        output_zip.unlink()

    with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in adapter_files:
            archive.write(path, arcname=path.name)

    with zipfile.ZipFile(output_zip) as archive:
        names = sorted(archive.namelist())

    for name in ["adapter_config.json", "adapter_model.safetensors"]:
        assert name in names, names
    print("wrote:", output_zip)
    print("zip contents:", names)


write_submission_zip(ADAPTER_DIR, SUBMISSION_ZIP_PATH)


## 12. Download

In [ ]:
files.download(str(SUBMISSION_ZIP_PATH))
